In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

coordinates = pd.read_csv(
    "results/galaxy_coordinates.csv.gz",
    dtype={"specObjID": "string"},
)

environment = pd.read_csv(
    "results/galaxy_environment.csv.gz",
    dtype={"specObjID": "string"},
)

galaxies = coordinates.merge(
    environment[["specObjID", "r10"]],
    on="specObjID",
    how="inner",
    validate="one_to_one",
)

if len(galaxies) != len(coordinates):
    raise ValueError("좌표와 이웃 거리 자료의 은하 ID 불일치")

In [2]:
VIEW_Z_MIN = 0.06
VIEW_Z_MAX = 0.08

view = galaxies.loc[
    galaxies["z"].between(
        VIEW_Z_MIN, VIEW_Z_MAX, inclusive="neither"
    )
    & np.isfinite(galaxies["r10"])
    & galaxies["r10"].gt(0)
]

plot_data = view.sample(
    n=min(50000, len(view)),
    random_state=42,
)

if plot_data.empty:
    raise ValueError("표시할 은하 없음")

len(plot_data)

50000

In [3]:
%matplotlib tk

In [ ]:
with plt.style.context("dark_background"):
    fig = plt.figure(figsize=(12, 8))
    ax = fig.add_subplot(111, projection="3d")

    points = ax.scatter(
        plot_data["X"],
        plot_data["Y"],
        plot_data["Z"],
        c=plot_data["r10"],
        cmap="plasma_r",
        norm=LogNorm(),
        s=3,
        alpha=0.8,
        linewidths=0,
        depthshade=False,
    )

    span = np.ptp(
        plot_data[["X", "Y", "Z"]].to_numpy(),
        axis=0,
    )
    ax.set_box_aspect(np.maximum(span, 1))

    ax.set_xlabel(r"$X\ [h^{-1}\mathrm{Mpc}]$")
    ax.set_ylabel(r"$Y\ [h^{-1}\mathrm{Mpc}]$")
    ax.set_zlabel(r"$Z\ [h^{-1}\mathrm{Mpc}]$")
    ax.set_title(
        f"Galaxy environment | {VIEW_Z_MIN} < z < {VIEW_Z_MAX}"
    )
    ax.view_init(elev=25, azim=125)

    fig.colorbar(
        points,
        ax=ax,
        label=r"$r_{10}\ [h^{-1}\mathrm{Mpc}]$",
        shrink=0.6,
        pad=0.1,
    )

    fig.savefig(
        "results/galaxies_3d_r10.png",
        dpi=200,
        bbox_inches="tight",
    )
    plt.show()

: 